In [1]:
import sys
import shutil
assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git

    # 2. Wejście do folderu projektu
    %cd ASR_project

    # 3. Zmiana gałęzi na 'PierwszyModel' (bo tam pracujesz)
    !git checkout SoftLabels

    !pip install torchmetrics

In [2]:
if not IS_COLAB: #na colabie to odwala
    %load_ext autoreload
    %autoreload 2

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.io import wavfile
from src.constants import Constants as C
from pathlib import Path
import torchmetrics

In [3]:
from src.augment import augment_audio, SpecAugment
from src.parsers import build_augmented_cache_soft, build_augmented_cache
from src.parsers import PhonemeWindowDataset
from src.parsers import DoubledSoftLabelCacheDataset, SoftLabelCacheDataset 
from src.NeuralModel import CRNN
from src.trainers import train_model, evaluate_tm, save_checkpoint, load_checkpoint

In [4]:
from src.trainers import SoftCrossEntropyLoss, SoftAccuracy

In [5]:
if (IS_COLAB):
    from google.colab import drive
    drive.mount('/content/drive')

Duze_dane = False

In [6]:
if(Duze_dane):
    # tutaj to gdy robimy duże dane
    DATA_DIR2 = "/content/dane_audio_lokalnie"
else:
    DATA_DIR2 = "../AutorskieDane/AutorskiDataset"

In [9]:
data_in_ram = None
# Wymuszamy budowanie w RAM, omijając uszkodzony cache na dysku
if True: 
    print('Budowanie cache w RAM (jednorazowe, to potrwa)...')
    data_in_ram = build_augmented_cache_soft(
        data_dir=DATA_DIR2,
        output_path=None,  # Nie zapisujemy na dysku
        n_augmentations=2, # tutaj trzeba przynajmniej 2, bo inaczej train set sie wywala, nie mialem czasu zeby to ogarnac
        standardize=True,
        silences_same=True,
        noiseprob=0.5,
        gainprob=0.5,
        tempo_prob=0.0,
        noise_level=(10, 30),
        gain_range=(-5, 5),
        tempo_range=(0.95, 1.05)
    )
    print('Cache wygenerowany w RAM!')

Budowanie cache w RAM (jednorazowe, to potrwa)...
znaleziono 46 plików
final: X=torch.Size([45027, 2, 128, 8]), y=torch.Size([45027, 39])
rozmiar: 0.38 GB
output_path=None — nie zapisuję na dysk, tylko zwracam
Cache wygenerowany w RAM!


In [10]:
if data_in_ram is not None:
    data = data_in_ram
device = 'cuda' if torch.cuda.is_available() else 'cpu'

X = data['X']           # (n_windows, n_aug, n_mels, win_frames)
y = data['y']
n_aug = data['n_aug']
print(f'Załadowano: X={X.shape}, y={y.shape}, n_aug={n_aug}')

Załadowano: X=torch.Size([45027, 2, 128, 8]), y=torch.Size([45027, 39]), n_aug=2


In [11]:
generator = torch.Generator().manual_seed(0)
indices = torch.randperm(len(X), generator=generator)
n_val = int(0.2 * len(X))
train_idx, val_idx = indices[n_val:], indices[:n_val]
train_set = DoubledSoftLabelCacheDataset(X[train_idx], y[train_idx], train=True, device=device)
val_set   = SoftLabelCacheDataset(X[val_idx],   y[val_idx],   train=False, device=device)



In [12]:
train_loader = DataLoader(train_set, batch_size=512, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_set,   batch_size=1024, shuffle=False,
                          num_workers=0, pin_memory=False)

In [13]:

model = CRNN().to(device)

# 7. Loss + metric — soft versions
crit = SoftCrossEntropyLoss(label_smoothing=0.05)
metric = SoftAccuracy(num_classes=C.N_CLASSES).to(device)
EPOCHS = 3
optim = torch.optim.NAdam(model.parameters(), lr=1e-5)
#mel_augmenter = SpecAugment(freq_mask_percent=0.2, time_mask_percent=0.125, p=0.5)
# 8. Trening
CHECKPOINT_PATH = "../trained_models/softALL.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN().to(device)
meta = load_checkpoint(CHECKPOINT_PATH, model, device=device)


In [14]:
history = train_model(
    model, optim, crit, metric,
    train_loader, val_loader,
    n_epochs=EPOCHS,
    patience=3, factor=0.5,
    device=device,
    early_stop_patience=8,
)

Epoch   1/3  train_loss=2.2780  train=50.53%  val=52.28%  lr=1.0e-05  ← best                    
Epoch   2/3  train_loss=2.2769  train=50.44%  val=52.33%  lr=1.0e-05  ← best                    
Epoch   3/3  train_loss=2.2775  train=50.40%  val=52.25%  lr=1.0e-05                    

Restored best model from epoch 2 (val=52.33%)


In [15]:
CHECKPOINT_PATH = "../trained_models/SoftFineTuned.pt"

config = {
    "sample_rate": C.SAMPLE_RATE,
    "n_fft": C.N_FFT,
    "hop_length": C.HOP_LENGTH,
    "n_mels": C.N_MELS,
    "win_frames": C.WIN_FRAMES,
    "shift_frames": C.SHIFT_FRAMES,
    "hidden": 64,
}

save_checkpoint(
    CHECKPOINT_PATH,
    model,
    labels=C.LABELS,
    config=config,
    history=history,
)
print(f"saved to {CHECKPOINT_PATH}")

if IS_COLAB:
    to_disc_path = '/content/ASR_project/SoftFineTuned.pt' # TU ZAWSZE NOWA NAZWA
    torch.save(model.state_dict(), to_disc_path)
    
    # Definiujemy ścieżkę do folderu
    sciezka_na_dysku = '/content/drive/MyDrive/asr_data/SoftFineTuned/'
    
    # 1. TWORZYMY FOLDER (jeśli go nie ma, zostanie stworzony; jeśli jest, kod pójdzie dalej bez błędu)
    os.makedirs(sciezka_na_dysku, exist_ok=True)
    
    # 2. Teraz kopiujemy plik
    shutil.copy(to_disc_path, sciezka_na_dysku)
    print("Model bezpiecznie skopiowany do nowego folderu na Dysk Google.")

saved to ../trained_models/SoftFineTuned.pt


In [26]:
# pierwsze 5 próbek
for i in range(67):
    sample_label = y[i]
    nonzero = sample_label.nonzero().squeeze(-1)
    proportions = sample_label[nonzero]
    print(f'sample {i}:')
    for j, prop in zip(nonzero, proportions):
        print(f'  {C.LABELS[j]:>5}: {prop:.2f}')
    print(f'  sum: {sample_label.sum():.3f}')
    print()

sample 0:
    sil: 1.00
  sum: 1.000

sample 1:
    sil: 1.00
  sum: 1.000

sample 2:
    sil: 1.00
  sum: 1.000

sample 3:
    sil: 1.00
  sum: 1.000

sample 4:
    sil: 1.00
  sum: 1.000

sample 5:
    sil: 1.00
  sum: 1.000

sample 6:
    sil: 1.00
  sum: 1.000

sample 7:
    sil: 1.00
  sum: 1.000

sample 8:
    sil: 1.00
  sum: 1.000

sample 9:
    sil: 1.00
  sum: 1.000

sample 10:
    sil: 1.00
  sum: 1.000

sample 11:
    sil: 1.00
  sum: 1.000

sample 12:
    sil: 1.00
  sum: 1.000

sample 13:
    sil: 1.00
  sum: 1.000

sample 14:
    sil: 1.00
  sum: 1.000

sample 15:
    sil: 1.00
  sum: 1.000

sample 16:
    sil: 1.00
  sum: 1.000

sample 17:
    sil: 1.00
  sum: 1.000

sample 18:
    sil: 1.00
  sum: 1.000

sample 19:
    sil: 1.00
  sum: 1.000

sample 20:
    sil: 1.00
  sum: 1.000

sample 21:
    sil: 1.00
  sum: 1.000

sample 22:
    sil: 1.00
  sum: 1.000

sample 23:
    sil: 1.00
  sum: 1.000

sample 24:
    sil: 1.00
  sum: 1.000

sample 25:
    sil: 1.00
  sum: 1.0